# Setup no Colab

Roda essas celulas em ordem num runtime com GPU (Runtime > Change runtime type > T4).
Nenhuma das nossas maquinas tem GPU NVIDIA, entao todo treino saiu daqui ou de um
kernel equivalente no Kaggle. Em CPU roda igual, so demora horas em vez de minutos.

In [ ]:
!nvidia-smi

## Clonar e instalar

In [ ]:
REPO = 'https://github.com/joaovtaf/deep-learning-pa1.git'

%cd /content
![ -d deep-learning-pa1 ] || git clone $REPO
%cd /content/deep-learning-pa1
!git pull --ff-only || true

In [ ]:
# o Colab ja vem com torch e cuda, entao so instala o que falta
!pip install -q albumentations scikit-image opencv-python-headless
import os, sys
os.environ['PYTHONPATH'] = '/content/deep-learning-pa1/src'
sys.path.insert(0, '/content/deep-learning-pa1/src')

In [ ]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

## Parte 0, teste unitario sintetico

Tem que rodar em menos de 5 minutos. Nao precisa baixar dado nenhum, o gerador
monta as elipses na hora.

In [ ]:
!python scripts/train.py --config configs/synthetic_baseline.yaml --device cuda
!python scripts/evaluate.py --config configs/synthetic_baseline.yaml \
    --checkpoint runs/synthetic_baseline/best.pt --device cuda

In [ ]:
# mesma coisa ja com a cabeca de instancias da Parte 2
!python scripts/train.py --config configs/synthetic_boundary.yaml --device cuda
!python scripts/evaluate.py --config configs/synthetic_boundary.yaml \
    --checkpoint runs/synthetic_boundary/best.pt --device cuda

## Dados

In [ ]:
!python scripts/download_dsb2018.py

## Parte 1, baseline binario

Segmentacao fundo vs objeto e instancias por limiar + componentes conexos.

In [ ]:
!python scripts/train.py --config configs/dsb2018_baseline.yaml --device cuda
!python scripts/evaluate.py --config configs/dsb2018_baseline.yaml \
    --checkpoint runs/dsb2018_baseline/best.pt --device cuda --panels 6

In [ ]:
# o item 4 pede a regra de matching explicita, entao mede as duas
!python scripts/evaluate.py --config configs/dsb2018_baseline.yaml \
    --checkpoint runs/dsb2018_baseline/best.pt --device cuda --matching hungarian \
    --out-dir runs/dsb2018_baseline/eval_test_hungarian --panels 0

## Parte 2, trilha A

Fronteira e watershed. A tabela de frequencia das classes sai antes porque e dela
que veio o alpha do config.

In [ ]:
!python scripts/class_stats.py --config configs/dsb2018_boundary.yaml
!python scripts/train.py --config configs/dsb2018_boundary.yaml --device cuda
!python scripts/evaluate.py --config configs/dsb2018_boundary.yaml \
    --checkpoint runs/dsb2018_boundary/best.pt --device cuda --panels 6

In [ ]:
# as mesmas metricas lado a lado com a baseline, que e o que a Parte 2 pede
!python scripts/compare.py \
    --baseline configs/dsb2018_baseline.yaml runs/dsb2018_baseline/best.pt \
    --instance configs/dsb2018_boundary.yaml runs/dsb2018_boundary/best.pt --device cuda

## Parte 3, ablacoes

Eixos 2 (funcao de perda) e 3 (contexto global), 2 seeds por configuracao. E a parte
mais cara: sao 6 configuracoes no eixo 2 e 3 no eixo 3, entao 18 treinos.

In [ ]:
!python scripts/ablations.py --config configs/dsb2018_boundary.yaml --axis 2 --seeds 0 1 --epochs 20 --device cuda

In [ ]:
!python scripts/ablations.py --config configs/dsb2018_boundary.yaml --axis 3 --seeds 0 1 --epochs 20 --device cuda

## Partes 4, 5 e 6

Essas nao treinam nada, so rodam inferencia com o checkpoint da Parte 2.

In [ ]:
!python scripts/part4_mosaic.py --config configs/dsb2018_boundary.yaml \
    --checkpoint runs/dsb2018_boundary/best.pt --device cuda

In [ ]:
!python scripts/part5_failures.py --config configs/dsb2018_boundary.yaml \
    --checkpoint runs/dsb2018_boundary/best.pt --device cuda

In [ ]:
!python scripts/part6_stress.py --config configs/dsb2018_boundary.yaml \
    --checkpoint runs/dsb2018_boundary/best.pt --device cuda

## Salvar os resultados no Drive

Senao perde tudo quando o runtime cai.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/pa1'
!cp -r runs '/content/drive/MyDrive/pa1/'